# Thời gian và tài nguyên parser

Đo riêng các bước gathering, parse, standardization, dedup và storage trên Colab CPU-only. Báo wall time, thời gian trung bình mỗi publication, peak RAM và dung lượng đầu ra kèm input/code revision. Thời gian tải corpus và cài môi trường được ghi riêng.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from crawler.reporting import load_m2, run_table
m2 = load_m2()
manifest, stats = m2['manifest'], m2['stats']
print('Input:', m2['directory'])
print('Phạm vi:', stats.get('publications'), 'paper;', stats.get('versions'), 'version')
print('Run:', stats.get('run_id'), '; status:', stats.get('status'))


In [ ]:
runs=run_table(m2['directory'])
display(runs)
display(pd.DataFrame({'metric':['final_output_bytes','avg_sec_per_processed_publication'],
                     'value':[stats.get('final_output_bytes'),stats.get('avg_sec_per_processed_publication')]}))
print('Source code hash:',stats.get('code_hash'))
print('Pipeline:',stats.get('pipeline_version'))
print('Root override hash:',stats.get('root_overrides_hash'))
print('Phương pháp validation:',stats.get('validation',{}).get('method'))
print('Validation có thể dùng đĩa tạm; disk peak đã lấy mẫu trước khi dọn index.')
if runs.empty or not runs['colab_cpu'].any():
    print('CHƯA CÓ benchmark Colab CPU-only. Các số đo trên chỉ thuộc run local đã nêu.')
if not runs.empty:
    runs.set_index('run_id')['runtime_sec'].plot.bar(title='Parser wall time từng run — kiểm tra số paper mới/dùng lại',ylabel='Giây')
    plt.tight_layout(); plt.show()


## Giới hạn

Run resume có tái sử dụng kết quả. Khi so sánh, kiểm tra source hash, version coverage và số paper xử lý mới. Dung lượng ba bảng chuẩn được tách khỏi working data/checkpoint.
